## sqlite分析数据

In [1]:
import pandas as pd
import sqlite3

sqlite准备工作

In [2]:
conn = sqlite3.connect('rental.db')

按区域统计租金均价、单价均价、房源数量

In [3]:
sql_region = '''
    select 区域,
        round(avg(租金),0) as 平均租金,
        round(avg(单价（每平米租金）),0) as 平均单价,
        count(*) as 房源数
    from rental
    group by 区域
    order by 平均租金 desc;
'''
df_region = pd.read_sql_query(sql_region, conn)
print(df_region)

     区域    平均租金   平均单价   房源数
0   西城区  6281.0  145.0  3509
1   东城区  6142.0  143.0  3509
2   海淀区  5381.0  118.0  3508
3   朝阳区  4710.0  107.0  3508
4   丰台区  3280.0   75.0  3508
5  石景山区  3220.0   73.0  3508
6   通州区  2686.0   59.0  3508
7   大兴区  2339.0   53.0  3508
8   昌平区  2300.0   53.0  3508
9   顺义区  2044.0   45.0  3508


从市中心向外围，租金和单价呈现明显的层层递减规律。想住核心区要承担高昂溢价，而选择外围近郊能省下将近 60%~70% 的租房成本。

带地铁与不带地铁的租金对比

In [4]:
sql_subway = '''
    select
        区域,
        case when 是否有地铁 == 1 then '近地铁' else '无地铁' end as 地铁情况,
        round(avg(租金),0) as 平均租金,
        count(*) as 房源数
    from rental
    group by 区域,地铁情况
'''
df_subway = pd.read_sql_query(sql_subway, conn)
print(df_subway)

      区域 地铁情况    平均租金   房源数
0    东城区  无地铁  6189.0  1409
1    东城区  近地铁  6110.0  2100
2    丰台区  无地铁  3225.0  1393
3    丰台区  近地铁  3316.0  2115
4    大兴区  无地铁  2277.0  1414
5    大兴区  近地铁  2381.0  2094
6    昌平区  无地铁  2379.0  1407
7    昌平区  近地铁  2247.0  2101
8    朝阳区  无地铁  4751.0  1389
9    朝阳区  近地铁  4684.0  2119
10   海淀区  无地铁  5402.0  1350
11   海淀区  近地铁  5368.0  2158
12  石景山区  无地铁  3221.0  1448
13  石景山区  近地铁  3219.0  2060
14   西城区  无地铁  6397.0  1421
15   西城区  近地铁  6202.0  2088
16   通州区  无地铁  2720.0  1467
17   通州区  近地铁  2661.0  2041
18   顺义区  无地铁  2103.0  1366
19   顺义区  近地铁  2007.0  2142


在本次统计中，近地铁的房租并没有比无地铁显著提高，核心区无地铁的房子凭借地段优势依旧比其他区域近地铁的房租贵。真正的分水岭在于地段而非是否近地铁。